# 讓 Agent 判斷什麼時候要叫工具

這章只看一件事：把工具 schema 交給模型，觀察模型是否回傳 tool call；應用層再檢查參數並執行真正的函式。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 填入模型連線設定

這章會真的呼叫 OpenAI-compatible 模型端點。請先把下面三個值換成你自己的設定：`API_KEY`、`BASE_URL`、`MODEL`。

In [ ]:
import json

from agentic_sdk import Workflow
from agentic_sdk.modules import PassThroughPerceive, ToolCallAction

API_KEY = '<要填入的 API_KEY>'
BASE_URL = '<要填入的 OpenAI-compatible BASE_URL>'
MODEL = '<要填入的支援 tool calling 的 MODEL>'

## 定義模型可以選擇的工具

這份 schema 會送進模型 API。支援 tool calling 的模型會根據使用者問題，決定是否回傳 `create_support_ticket` 這個 tool call。

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'create_support_ticket',
            'description': '建立客服處理單。',
            'parameters': {
                'type': 'object',
                'properties': {
                    'title': {'type': 'string', 'description': '處理單標題'},
                    'priority': {'type': 'string', 'description': '優先度：low、normal 或 high'},
                },
                'required': ['title', 'priority'],
                'additionalProperties': False,
            },
        },
    }
]

tools

## 準備真正會做事的函式

模型只提出工具名稱與參數；真正建立處理單的是這個函式。正式系統裡，這裡通常會換成資料庫、後端 API 或內部服務。

In [ ]:
def create_support_ticket(title, priority):
    if priority not in {'low', 'normal', 'high'}:
        raise ValueError('priority 必須是 low、normal 或 high')
    return {
        'ticket_id': 'TCK-1001',
        'title': title,
        'priority': priority,
        'status': 'created',
    }

## 建立會呼叫模型的 workflow

`ToolCallAction` 會把使用者問題、工具 schema、`tool_choice='auto'` 一起送到模型端點。模型是否回傳 tool call，取決於端點是否支援工具呼叫，以及模型是否判斷這題需要工具。

In [ ]:
workflow = Workflow(
    workflow_name='Tool call demo Agent',
    perceive=PassThroughPerceive(),
    action=ToolCallAction(
        api_key=API_KEY,
        base_url=BASE_URL,
        model=MODEL,
        tools=tools,
        tool_choice='auto',
    ),
)

## 執行一次，查看模型是否回傳 tool call

這一步會真的呼叫模型。`latest_tool_calls` 不是手寫結果，而是 `ToolCallAction` 從模型回應整理出的資料。

In [ ]:
result = workflow.run('AI Hub 無法登入，請幫我開高優先度處理單。')
latest_tool_calls = result.entities.get('latest_tool_calls') or []

print(result.final_message)
print(latest_tool_calls)

if not latest_tool_calls:
    raise RuntimeError('模型沒有回傳 tool call；請確認 MODEL 支援工具呼叫，或調整使用者問題。')

## 檢查參數，再由應用層執行工具

模型回傳的參數不能直接相信。先檢查工具名稱、JSON 內容與必要欄位，再呼叫真正的函式。

In [ ]:
allowed_tools = {'create_support_ticket': create_support_ticket}
required_arguments = {'title', 'priority'}

call = latest_tool_calls[0]
function = call['function']
function_name = function['name']
arguments = json.loads(function['arguments'])

if function_name not in allowed_tools:
    raise ValueError(f'不允許的工具：{function_name}')
missing = required_arguments - set(arguments)
if missing:
    raise ValueError(f'缺少必要參數：{sorted(missing)}')

tool_result = allowed_tools[function_name](**arguments)
print(tool_result)